In [6]:
from pathlib import Path

DATA_ROOT = Path("../data/raw/fi2010/BenchmarkDatasets")


## Load one FI-2010 training file

The FI-2010 files are stored as numerical matrices. We first load one cross-validation fold to inspect its orientation and dimensions.

In [9]:
import numpy as np

zscore_dir = DATA_ROOT / "NoAuction" / "1.NoAuction_Zscore"

train_dir = zscore_dir / "NoAuction_Zscore_Training"
train_file = train_dir / "Train_Dst_NoAuction_ZScore_CF_1.txt"

test_dir = zscore_dir / "NoAuction_Zscore_Testing"
test_file = test_dir / "Test_Dst_NoAuction_Zscore_CF_1.txt"

raw_train = np.loadtxt(train_file)
raw_test = np.loadtxt(test_file)

In [11]:
train_data = raw_train.T
test_data = raw_test.T

X_train = train_data[:, :144]
y_train_all = train_data[:, 144:]

X_test = test_data[:, :144]
y_test_all = test_data[:, 144:]

print("X_train:", X_train.shape)
print("y_train_all:", y_train_all.shape)
print("X_test:", X_test.shape)
print("y_test_all:", y_test_all.shape)

X_train: (39512, 144)
y_train_all: (39512, 5)
X_test: (38397, 144)
y_test_all: (38397, 5)


In [12]:
target_col = 3

y_train = y_train_all[:, target_col].astype(int)
y_test = y_test_all[:, target_col].astype(int)

print("Train target distribution:")
values, counts = np.unique(y_train, return_counts=True)
for value, count in zip(values, counts):
    print(f"class {value}: {count} ({count / len(y_train):.2%})")

print("\nTest target distribution:")
values, counts = np.unique(y_test, return_counts=True)
for value, count in zip(values, counts):
    print(f"class {value}: {count} ({count / len(y_test):.2%})")

Train target distribution:
class 1: 13349 (33.78%)
class 2: 13821 (34.98%)
class 3: 12342 (31.24%)

Test target distribution:
class 1: 13764 (35.85%)
class 2: 11749 (30.60%)
class 3: 12884 (33.55%)


## Majority-class baseline

The majority-class baseline always predicts the most common class in the training data. For the selected target horizon, this gives about 30.6% test accuracy and a macro F1 of 0.156. Any useful model should improve on both metrics and should predict all three classes.

In [13]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

majority_class = np.bincount(y_train).argmax()

y_pred_majority = np.full_like(y_test, fill_value=majority_class)

print("Majority class:", majority_class)
print("Accuracy:", accuracy_score(y_test, y_pred_majority))
print("Macro F1:", f1_score(y_test, y_pred_majority, average="macro"))

print(classification_report(y_test, y_pred_majority))

Majority class: 2
Accuracy: 0.30598744693595853
Macro F1: 0.15619723740012498
              precision    recall  f1-score   support

           1       0.00      0.00      0.00     13764
           2       0.31      1.00      0.47     11749
           3       0.00      0.00      0.00     12884

    accuracy                           0.31     38397
   macro avg       0.10      0.33      0.16     38397
weighted avg       0.09      0.31      0.14     38397



/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_d

## First linear baseline
We train a multinomial logistic regression model as the first real predictive baseline. This checks whether the FI-2010 feature matrix contains signal beyond the majority-class baseline.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )
)

logreg.fit(X_train, y_train)

y_pred_logreg = logreg.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_logreg))
print("Macro F1:", f1_score(y_test, y_pred_logreg, average="macro"))
print(classification_report(y_test, y_pred_logreg))

/Users/sohamaggarwal/Desktop/limit-order-book-forecasting/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Accuracy: 0.4393311977498242
Macro F1: 0.4306110349155696
              precision    recall  f1-score   support

           1       0.49      0.40      0.44     13764
           2       0.41      0.32      0.36     11749
           3       0.42      0.59      0.49     12884

    accuracy                           0.44     38397
   macro avg       0.44      0.44      0.43     38397
weighted avg       0.44      0.44      0.43     38397



## Observation

The majority-class baseline achieved 30.6% accuracy and 0.156 macro F1. A multinomial logistic regression model improved performance to 43.9% accuracy and 0.431 macro F1 on the official test split for label horizon 3.

This suggests that the FI-2010 feature matrix contains predictive signal beyond class imbalance. The model predicts all three classes, although performance is weakest for the stationary class. This motivates testing stronger nonlinear models and later analyzing which microstructure features contribute most to predictive performance.